# A2A Planner Client

Talks to `planner_agent.py` (LangChain agent + human-in-the-loop) over A2A
and streams the reply.

```
  You                 Client                        Planner
   |                    |                              |
   | "plan Kyoto"       |                              |
   |------------------->|  message/stream  ----------->|
   |                    |<-- task (save task/context) -|
   |<-- live tokens ----|<-- artifact "response" ------|
   |<-- "How many days?"|<-- status input-required ----|
   |                    |                              |
   | "3 days"           |                              |
   |------------------->|  SAME task_id + context_id ->|
   |<-- draft plan -----|<-- status input-required ----|
   |                    |                              |
   | "approve"          |                              |
   |------------------->|  SAME task_id + context_id ->|
   |<-- itinerary ------|<-- artifact "itinerary" -----|
   |<-- done -----------|<-- status completed ---------|
```

**Ids**

| last task state | what the next message sends |
|---|---|
| `input-required` | `task_id` + `context_id` (continues the SAME task) |
| `completed` | `context_id` only (a finished task cannot take messages, so the server opens a new one) |

**What this notebook does NOT do**

- No `tasks/cancel`. Interrupting a cell only stops the client side.
- No reconnect if the stream drops mid-task.
- Text parts only. File and data parts are ignored.

**Before running:** start the agent in a terminal.

```
python planner_agent.py
```


## 1. Imports and config

In [1]:
import uuid

import httpx

from a2a.client import A2ACardResolver, ClientConfig, create_client
from a2a.types import Message, Part, Role, SendMessageRequest, TaskState

AGENT_URL = "http://localhost:9103"

## 2. Helpers

`_request` builds one user message.

```
task_id     set only when continuing an input-required task
context_id  set whenever we have one (keeps the conversation grouped)
```


In [2]:
def _text(parts) -> str:
    """Join all text parts. Non-text parts are ignored."""
    return "".join(part.text for part in parts if part.text)


def _request(text, task_id, context_id) -> SendMessageRequest:
    message = Message(
        message_id=str(uuid.uuid4()),
        role=Role.ROLE_USER,
        parts=[Part(text=text)],
    )

    # Empty string means "not set" for protobuf string fields.
    if task_id:
        message.task_id = task_id
    if context_id:
        message.context_id = context_id

    return SendMessageRequest(message=message)

## 3. One turn

Sends one message and prints the stream as it arrives.

```
task            -> remember task_id / context_id
artifact_update
    "response"  -> LLM tokens, printed inline
    "itinerary" -> approved plan, printed as a block
status_update
    working          -> progress line
    input-required   -> the question or the draft plan
    completed        -> done
    failed/canceled  -> reason
```


In [3]:
async def send_turn(client, text, task_id, context_id):

    final_state = None

    # True while "response" tokens are being printed on one line.
    streaming_line = False

    async for event in client.send_message(_request(text, task_id, context_id)):

        kind = event.WhichOneof("payload")

        # STEP 1  Task created -> remember its ids.
        if kind == "task":
            task_id = event.task.id
            context_id = event.task.context_id

        # STEP 2  Artifact chunks.
        elif kind == "artifact_update":
            update = event.artifact_update
            chunk = _text(update.artifact.parts)

            if update.artifact.name == "response":
                if chunk:
                    if not streaming_line:
                        print("\nPlanner: ", end="")
                        streaming_line = True
                    print(chunk, end="", flush=True)
                if update.last_chunk and streaming_line:
                    print()
                    streaming_line = False

            elif update.artifact.name == "itinerary":
                print("\n" + "=" * 60)
                print("ITINERARY")
                print("=" * 60)
                print(chunk)
                print("=" * 60)

        # STEP 3  Status updates.
        elif kind == "status_update":
            status = event.status_update.status
            final_state = status.state
            note = _text(status.message.parts)

            # Break the token line so status text doesn't glue onto it.
            if note and streaming_line:
                print()
                streaming_line = False

            if final_state == TaskState.TASK_STATE_WORKING:
                if note:
                    print(f"  [{note}]")

            elif final_state == TaskState.TASK_STATE_INPUT_REQUIRED:
                print(f"\nPlanner: {note}")

            elif final_state == TaskState.TASK_STATE_COMPLETED:
                print(f"\n[completed] {note}")

            else:
                print(f"\n[{TaskState.Name(final_state)}] {note}")

        # STEP 4  Plain message reply (agent answered without a task).
        elif kind == "message":
            print(f"\nPlanner: {_text(event.message.parts)}")

    return task_id, context_id, final_state

## 4. Connect

Fetches `/.well-known/agent-card.json` and opens a streaming client.
`http` stays open for the rest of the notebook, so run the last cell
when you are done.


In [4]:
http = httpx.AsyncClient(timeout=120)

card = await A2ACardResolver(http, AGENT_URL).get_agent_card()
print(f"{card.name} - {card.description}")

client = await create_client(
    card,
    client_config=ClientConfig(httpx_client=http, streaming=True),
)

# Conversation state, updated by chat() below.
TASK_ID = None
CONTEXT_ID = None
STATE = None

planner - Plans a trip with an LLM. Asks for missing details and asks for approval before publishing the plan.


## 5. chat()

Wraps `send_turn` and keeps the ids between cells, so each cell is
one turn of the conversation.


In [5]:
async def chat(text):
    global TASK_ID, CONTEXT_ID, STATE

    print(f"You: {text}")

    # Continue the same task only if it is waiting for us.
    continue_task = STATE == TaskState.TASK_STATE_INPUT_REQUIRED

    TASK_ID, CONTEXT_ID, STATE = await send_turn(
        client,
        text,
        TASK_ID if continue_task else None,
        CONTEXT_ID,
    )

## 6. Turn 1: no duration -> the agent asks (`input-required`)

In [6]:
await chat("plan a trip to Kyoto")

You: plan a trip to Kyoto
  [Planning your trip...]

Planner: How many days will your Kyoto trip be?


## 7. Turn 2: answer it -> the agent drafts a plan and asks for approval

In [7]:
await chat("3 days")

You: 3 days
  [Planning your trip...]

Planner: Here is the draft plan:

Day 1:
- Morning: Visit Fushimi Inari Taisha, famous for its thousands of vermilion torii gates. Enjoy an early hike up the mountain trail for great views.
- Afternoon: Explore Kiyomizu-dera Temple and stroll through the historic streets of Higashiyama District (Sannenzaka and Ninenzaka).
- Evening: Wander through Gion district, Kyoto's famous geisha quarter, and enjoy a traditional kaiseki dinner.

Day 2:
- Morning: Visit Kinkaku-ji (Golden Pavilion), one of Kyoto's most iconic sites, surrounded by beautiful gardens.
- Afternoon: Explore Ryoan-ji Temple's famous rock garden, then head to Arashiyama to walk through the Bamboo Grove.
- Evening: Enjoy a relaxing boat ride on the Hozu River or visit Arashiyama's Monkey Park for sunset views over Kyoto.

Day 3:
- Morning: Explore Nijo Castle, a UNESCO World Heritage site with beautiful gardens and the famous "nightingale floors."
- Afternoon: Visit Nishiki Market for 

## 8. Turn 3: reject with feedback

Anything that is not `approve` / `yes` / `ok` is treated as feedback.
The agent revises and asks again.


In [8]:
await chat("make day 1 more relaxed")

You: make day 1 more relaxed
  [Planning your trip...]

Planner: Here is the draft plan:

Day 1:
- Morning: Start with a leisurely breakfast at a local café, then take a relaxed walk through the Higashiyama District, admiring the historic streets of Sannenzaka and Ninenzaka at an easy pace.
- Afternoon: Visit Kiyomizu-dera Temple, taking time to enjoy the peaceful grounds and scenic views without rushing. Stop for tea or a light snack nearby.
- Evening: Take a gentle evening stroll through the Gion district to see the lantern-lit streets, followed by a relaxed traditional dinner at a cozy local restaurant.

Day 2:
- Morning: Visit Kinkaku-ji (Golden Pavilion), one of Kyoto's most iconic sites, surrounded by beautiful gardens.
- Afternoon: Explore Ryoan-ji Temple's famous rock garden, then head to Arashiyama to walk through the Bamboo Grove.
- Evening: Enjoy a relaxing boat ride on the Hozu River or visit Arashiyama's Monkey Park for sunset views over Kyoto.

Day 3:
- Morning: Explore N

## 9. Turn 4: approve -> the itinerary artifact arrives, task completes

In [9]:
await chat("approve")

You: approve
  [Planning your trip...]
  [Running publish_itinerary...]

ITINERARY
Day 1:
- Morning: Start with a leisurely breakfast at a local café, then take a relaxed walk through the Higashiyama District, admiring the historic streets of Sannenzaka and Ninenzaka at an easy pace.
- Afternoon: Visit Kiyomizu-dera Temple, taking time to enjoy the peaceful grounds and scenic views without rushing. Stop for tea or a light snack nearby.
- Evening: Take a gentle evening stroll through the Gion district to see the lantern-lit streets, followed by a relaxed traditional dinner at a cozy local restaurant.

Day 2:
- Morning: Visit Kinkaku-ji (Golden Pavilion), one of Kyoto's most iconic sites, surrounded by beautiful gardens.
- Afternoon: Explore Ryoan-ji Temple's famous rock garden, then head to Arashiyama to walk through the Bamboo Grove.
- Evening: Enjoy a relaxing boat ride on the Hozu River or visit Arashiyama's Monkey Park for sunset views over Kyoto.

Day 3:
- Morning: Explore Nijo Cas

## 10. Free-form loop (optional)

Same thing as a prompt loop. Type `exit` to stop.


In [10]:
# while True:
#     text = input("You: ").strip()
#     if text.lower() in {"exit", "quit"}:
#         break
#     if text:
#         await chat(text)

## 12. Close the HTTP client

## 11. The full task as JSON

`tasks/get` returns the server's stored task: status, every artifact,
and the whole message history.

```
task.id          stable across the whole conversation above
task.context_id  groups tasks of one conversation
task.status      final state + last message
task.artifacts   "response" chunks joined + "itinerary"
task.history     every user and agent message
```


In [11]:
import json

from a2a.types import GetTaskRequest
from google.protobuf.json_format import MessageToDict

task = await client.get_task(GetTaskRequest(id=TASK_ID))

print(json.dumps(MessageToDict(task), indent=2))

{
  "id": "baf1946b-9bc6-4ff9-b37a-c3fc67905328",
  "contextId": "802d5daf-c344-468a-9f93-c54da75d33e4",
  "status": {
    "state": "TASK_STATE_COMPLETED",
    "message": {
      "messageId": "28fbfb0e-d9db-4d3b-bf25-7cfdd5f79016",
      "role": "ROLE_AGENT",
      "parts": [
        {
          "text": "Trip plan completed."
        }
      ]
    },
    "timestamp": "2026-09-23T03:36:43.216725Z"
  },
  "artifacts": [
    {
      "artifactId": "4b7ef2a1-b998-4cd7-8dfd-0eeead6e3418",
      "name": "itinerary",
      "parts": [
        {
          "text": "Day 1:\n- Morning: Start with a leisurely breakfast at a local caf\u00e9, then take a relaxed walk through the Higashiyama District, admiring the historic streets of Sannenzaka and Ninenzaka at an easy pace.\n- Afternoon: Visit Kiyomizu-dera Temple, taking time to enjoy the peaceful grounds and scenic views without rushing. Stop for tea or a light snack nearby.\n- Evening: Take a gentle evening stroll through the Gion district to see t

In [12]:
await http.aclose()